Helper Functions

In [2]:
import pickle
from pathlib import Path

CACHE_PATH = Path("../data/document_groups.pkl")

def load_document_groups():
    with open(CACHE_PATH, "rb") as f:
        return pickle.load(f)


In [3]:
data = load_document_groups()
print(f"Loaded {len(data)} applicants")

Loaded 4 applicants


In [5]:
print(type(data))

<class 'list'>


In [4]:
import fitz
from PIL import Image
import io

def pdf_bytes_to_images(pdf_bytes: bytes, dpi: int = 300):
    images = []
    with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
        zoom = dpi / 72
        matrix = fitz.Matrix(zoom, zoom)
        for i in range(len(doc)):
            pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
            img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
            images.append(img)
    return images



In [ ]:
def get_all_applicant_images(dpi=300):
    groups = load_document_groups()

    for group in groups:
        applicant_id = group["applicant_id"]
        print(f"Processing Applicant {applicant_id}")

        for doc in group["documents"]:
            filename = doc["filename"].lower()
            if not filename.endswith(".pdf"):
                continue

            try:
                pdf_bytes = doc["content"]
                images = pdf_bytes_to_images(pdf_bytes, dpi=dpi)
                # yield each page to the model
                for img in images:
                    yield applicant_id, filename, img

            except Exception as e:
                print(f"Failed: {filename}: {e}")


Read in PDFs and convert to images

In [ ]:
def load_all_applicant_images():
    initialize_applicant_mapping()  # Ensures global data loaded

    all_images = {}  # { applicant_id: [images...] }

    for group in _all_document_groups:
        applicant_id = group["applicant_id"]

        applicant_images = []
        for doc in group["documents"]:
            if doc["filename"].lower().endswith(".pdf"):
                try:
                    imgs = pdf_bytes_to_images(doc["content"])
                    applicant_images.extend(imgs)
                except Exception as e:
                    print(f"PDF conversion failed ({doc['filename']}): {e}")

        all_images[applicant_id] = applicant_images

    return all_images
